# GLM-5 / Kimi Distillation: Eval & Publish

This notebook runs on **Colab Pro** (A100/V100) for fast local inference.

Steps:
1. Download LoRA weights from Tinker
2. Merge with base Qwen3.5-4B
3. Evaluate all models on clean benchmarks (GSM8K, MATH, ARC, MMLU-Pro)
4. Side-by-side qualitative comparison on trick questions
5. Push merged models to HuggingFace
6. Export GGUF for local inference

In [ ]:
# Step 0: Install deps + mount Drive for checkpoints
!pip install unsloth transformers datasets huggingface_hub tinker

from google.colab import drive, userdata
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/distillreasoning'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive dir: {DRIVE_DIR}')

In [ ]:
# Step 1: Download LoRA weights from Tinker
import tinker
import requests
import zipfile
from pathlib import Path

os.environ['TINKER_API_KEY'] = userdata.get('TINKER_API_KEY')

service = tinker.ServiceClient()
rest = service.create_rest_client()

# Our 3 SFT checkpoints (update these after GRPO with the GRPO checkpoint paths)
CHECKPOINTS = {
    'glm5': 'tinker://0fbca836-2aae-5500-b28d-93c2a46a328b:train:0/sampler_weights/qwen35-4b-glm5-final',
    'kimi': 'tinker://f5795e66-71e4-5cf4-9ebe-1cc14c27aa6e:train:0/sampler_weights/qwen35-4b-kimi-final',
    'combined': 'tinker://41e7bd9e-e49f-5f13-a5ff-f4339faab448:train:0/sampler_weights/qwen35-4b-combined-final',
}

for name, tinker_path in CHECKPOINTS.items():
    print(f'Downloading {name}...')
    url_resp = rest.get_checkpoint_archive_url_from_tinker_path(tinker_path).result()
    url = url_resp.url
    
    # Download archive
    local_zip = f'/tmp/lora_{name}.zip'
    r = requests.get(url, stream=True)
    with open(local_zip, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    
    # Extract
    extract_dir = f'{DRIVE_DIR}/lora_{name}'
    with zipfile.ZipFile(local_zip) as z:
        z.extractall(extract_dir)
    print(f'  Extracted to {extract_dir}')
    print(f'  Files: {os.listdir(extract_dir)}')

print('\nAll LoRA weights downloaded!')

In [ ]:
# Step 2: Merge LoRA with base model using Unsloth
from unsloth import FastLanguageModel
import torch

BASE_MODEL = 'Qwen/Qwen3.5-4B'
max_seq_length = 4096

merged_models = {}
for name in ['glm5', 'kimi', 'combined']:
    print(f'\nMerging {name}...')
    lora_dir = f'{DRIVE_DIR}/lora_{name}'
    
    # Load base + LoRA
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=lora_dir,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=False,  # Load in full precision for merging
    )
    
    # Save merged model
    merged_dir = f'{DRIVE_DIR}/merged_{name}'
    model.save_pretrained_merged(merged_dir, tokenizer, save_method='merged_16bit')
    merged_models[name] = merged_dir
    print(f'  Saved merged model to {merged_dir}')

print('\nAll models merged!')

In [ ]:
# Step 3: Evaluate all models on clean benchmarks
# This is the FAST version — local GPU inference, no Tinker API calls

import json, re, random, time
from datasets import load_dataset

random.seed(999)  # Same eval seed as Tinker evals

SYSTEM_MSG = 'You are a helpful reasoning assistant. Think through problems step by step before answering.'

def extract_final_number(text):
    text_clean = text.replace(',', '')
    answer_patterns = [
        r'(?:the\s+)?(?:final\s+)?answer\s+is\s*:?\s*\$?\s*(-?\d+\.?\d*)',
        r'(?:therefore|thus|so)\s*,?\s*.*?(?:is|=|equals)\s*\$?\s*(-?\d+\.?\d*)\s*(?:\\.|$|\n)',
        r'\\boxed\{(-?\d+\.?\d*)\}',
        r'\*\*\$?(-?\d+\.?\d*)\$?\*\*\s*(?:\\.|$|\n|<)',
    ]
    for pat in answer_patterns:
        matches = re.findall(pat, text_clean, re.IGNORECASE)
        if matches:
            return float(matches[-1])
    numbers = re.findall(r'-?\d+\.?\d*', text_clean)
    return float(numbers[-1]) if numbers else None

def extract_boxed(text):
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    return match.group(1).strip() if match else None

def generate(model, tokenizer, problem, max_new_tokens=2048):
    messages = [
        {'role': 'system', 'content': SYSTEM_MSG},
        {'role': 'user', 'content': problem},
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to('cuda')
    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=max_new_tokens, temperature=0.6, top_p=0.95, do_sample=True)
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

def eval_gsm8k(model, tokenizer, n=500):
    ds = load_dataset('openai/gsm8k', 'main', split='train')  # Train split — clean!
    rng = random.Random(999)
    indices = rng.sample(range(len(ds)), n)
    correct = 0
    for idx, i in enumerate(indices):
        row = ds[i]
        response = generate(model, tokenizer, row['question'])
        expected = row['answer'].split('####')[-1].strip()
        try:
            exp_num = float(expected.replace(',', ''))
            mod_num = extract_final_number(response)
            if mod_num is not None and abs(mod_num - exp_num) < 0.01:
                correct += 1
        except ValueError:
            pass
        if (idx + 1) % 50 == 0:
            print(f'  GSM8K {idx+1}/{n}: {correct}/{idx+1} ({100*correct/(idx+1):.1f}%)')
    return {'benchmark': 'GSM8K', 'accuracy': round(100 * correct / n, 1), 'correct': correct, 'n': n}

def eval_math(model, tokenizer, n=500):
    ds = load_dataset('SuperSecureHuman/competition_math_hf_dataset', split='test')
    rng = random.Random(999)
    indices = rng.sample(range(len(ds)), n)
    correct = 0
    for idx, i in enumerate(indices):
        row = ds[i]
        response = generate(model, tokenizer, row['problem'])
        expected_boxed = extract_boxed(row.get('solution', ''))
        model_boxed = extract_boxed(response)
        if expected_boxed and model_boxed and expected_boxed.strip() == model_boxed.strip():
            correct += 1
        if (idx + 1) % 50 == 0:
            print(f'  MATH {idx+1}/{n}: {correct}/{idx+1} ({100*correct/(idx+1):.1f}%)')
    return {'benchmark': 'MATH', 'accuracy': round(100 * correct / n, 1), 'correct': correct, 'n': n}

def eval_arc(model, tokenizer, n=500):
    ds = load_dataset('allenai/ai2_arc', 'ARC-Challenge', split='test')
    rng = random.Random(999)
    indices = rng.sample(range(len(ds)), n)
    correct = 0
    for idx, i in enumerate(indices):
        row = ds[i]
        choices = row['choices']
        choice_text = '\n'.join(f'{l}. {t}' for l, t in zip(choices['label'], choices['text']))
        problem = f"{row['question']}\n\nChoices:\n{choice_text}"
        expected = row['answerKey']
        response = generate(model, tokenizer, problem)
        resp_upper = response.upper()
        found = False
        for pat in [rf'answer\s+is\s+\(?{expected}\)?', rf'\({expected}\)', rf'\b{expected}\b\s*[\\.)\]]']:
            if re.search(pat, resp_upper):
                found = True
                break
        if not found:
            last_caps = re.findall(r'\b([A-E])\b', resp_upper)
            if last_caps and last_caps[-1] == expected:
                found = True
        if found:
            correct += 1
        if (idx + 1) % 50 == 0:
            print(f'  ARC {idx+1}/{n}: {correct}/{idx+1} ({100*correct/(idx+1):.1f}%)')
    return {'benchmark': 'ARC', 'accuracy': round(100 * correct / n, 1), 'correct': correct, 'n': n}

print('Eval functions ready. Will run in step 3b.')

In [ ]:
# Step 3b: Run evals on all models
# Load each model, eval, free memory, next

all_results = {}

models_to_eval = [
    ('base-4b', BASE_MODEL, None),
    ('distilled-glm5', f'{DRIVE_DIR}/merged_glm5', None),
    ('distilled-kimi', f'{DRIVE_DIR}/merged_kimi', None),
    ('distilled-combined', f'{DRIVE_DIR}/merged_combined', None),
]

for model_name, model_path, _ in models_to_eval:
    print(f'\n{"="*60}')
    print(f'Evaluating: {model_name}')
    print(f'{"="*60}')
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    
    results = {'model': model_name}
    for bench_name, bench_fn in [('gsm8k', eval_gsm8k), ('math', eval_math), ('arc', eval_arc)]:
        print(f'\n  Running {bench_name}...')
        results[bench_name] = bench_fn(model, tokenizer)
        print(f'  {bench_name}: {results[bench_name]["accuracy"]}%')
    
    all_results[model_name] = results
    
    # Free GPU memory
    del model, tokenizer
    torch.cuda.empty_cache()

# Print comparison table
print(f'\n{"="*60}')
print('FINAL RESULTS')
print(f'{"="*60}')
print(f'{"Model":25s} {"GSM8K":>8s} {"MATH":>8s} {"ARC":>8s}')
print('-' * 55)
for name, r in all_results.items():
    g = r.get('gsm8k', {}).get('accuracy', '—')
    m = r.get('math', {}).get('accuracy', '—')
    a = r.get('arc', {}).get('accuracy', '—')
    print(f'{name:25s} {str(g)+"%":>8s} {str(m)+"%":>8s} {str(a)+"%":>8s}')

# Save to Drive
with open(f'{DRIVE_DIR}/eval_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'\nResults saved to {DRIVE_DIR}/eval_results.json')

In [ ]:
# Step 3c: Few-shot evaluation (for comparability with published benchmarks)
#
# Published benchmarks typically use 5-shot (GSM8K) or 25-shot (ARC) prompting.
# Our zero-shot numbers are internally consistent but not comparable to model cards.
# This cell adds few-shot eval so we can compare against published numbers.

GSM8K_FEW_SHOT_EXAMPLES = [
    {"q": "There are 15 trees in the grove. Grove workers will plant trees in the grove today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?",
     "a": "There are 15 trees originally. Then there were 21 trees after some more were planted. So there must have been 21 - 15 = 6. #### 6"},
    {"q": "If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?",
     "a": "There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5. #### 5"},
    {"q": "Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?",
     "a": "Originally, Leah had 32 chocolates. Her sister had 42. So in total they had 32 + 42 = 74. After eating 35, they had 74 - 35 = 39. #### 39"},
    {"q": "Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?",
     "a": "Jason started with 20 lollipops. Then he had 12 after giving some to Denny. So he gave Denny 20 - 12 = 8. #### 8"},
    {"q": "Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?",
     "a": "Shawn started with 5 toys. If he got 2 toys each from mom and dad, then that is 4 more toys. 5 + 4 = 9. #### 9"},
]

def generate_few_shot(model, tokenizer, problem, examples, max_new_tokens=2048):
    """Generate with few-shot examples prepended as user/assistant turns."""
    messages = [{'role': 'system', 'content': SYSTEM_MSG}]
    for ex in examples:
        messages.append({'role': 'user', 'content': ex['q']})
        messages.append({'role': 'assistant', 'content': ex['a']})
    messages.append({'role': 'user', 'content': problem})
    
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to('cuda')
    with torch.no_grad():
        outputs = model.generate(inputs, max_new_tokens=max_new_tokens, temperature=0.6, top_p=0.95, do_sample=True)
    return tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)

def eval_gsm8k_few_shot(model, tokenizer, n=500, n_shot=5):
    """GSM8K with few-shot examples (comparable to published benchmarks)."""
    ds = load_dataset('openai/gsm8k', 'main', split='train')
    rng = random.Random(999)
    indices = rng.sample(range(len(ds)), n)
    examples = GSM8K_FEW_SHOT_EXAMPLES[:n_shot]
    correct = 0
    for idx, i in enumerate(indices):
        row = ds[i]
        response = generate_few_shot(model, tokenizer, row['question'], examples)
        expected = row['answer'].split('####')[-1].strip()
        try:
            exp_num = float(expected.replace(',', ''))
            mod_num = extract_final_number(response)
            if mod_num is not None and abs(mod_num - exp_num) < 0.01:
                correct += 1
        except ValueError:
            pass
        if (idx + 1) % 50 == 0:
            print(f'  GSM8K {n_shot}-shot {idx+1}/{n}: {correct}/{idx+1} ({100*correct/(idx+1):.1f}%)')
    return {'benchmark': f'GSM8K ({n_shot}-shot)', 'accuracy': round(100 * correct / n, 1), 'correct': correct, 'n': n}

def eval_gpqa_diamond(model, tokenizer, n=198):
    """GPQA Diamond — graduate-level science. For Jackrong comparison."""
    ds = load_dataset('Idavidrein/gpqa', 'gpqa_diamond', split='train')
    n = min(n, len(ds))
    rng = random.Random(999)
    indices = rng.sample(range(len(ds)), n)
    correct = 0
    for idx, i in enumerate(indices):
        row = ds[i]
        # Format as multiple choice
        choices = [row.get(f'choice_{c}', '') for c in ['a', 'b', 'c', 'd'] if row.get(f'choice_{c}')]
        letters = [chr(65 + j) for j in range(len(choices))]
        choice_text = '\n'.join(f'{l}. {c}' for l, c in zip(letters, choices))
        problem = f"{row['question']}\n\nChoices:\n{choice_text}"
        
        response = generate(model, tokenizer, problem)
        expected = row.get('correct_answer', '')
        
        resp_upper = response.upper()
        found = False
        for pat in [rf'answer\s+is\s+\(?{expected}\)?', rf'\({expected}\)', rf'\b{expected}\b\s*[\\.)\]]']:
            if re.search(pat, resp_upper):
                found = True
                break
        if not found:
            last_caps = re.findall(r'\b([A-D])\b', resp_upper)
            if last_caps and last_caps[-1] == expected:
                found = True
        if found:
            correct += 1
        if (idx + 1) % 50 == 0:
            print(f'  GPQA Diamond {idx+1}/{n}: {correct}/{idx+1} ({100*correct/(idx+1):.1f}%)')
    return {'benchmark': 'GPQA Diamond (0-shot)', 'accuracy': round(100 * correct / n, 1), 'correct': correct, 'n': n}

print('Few-shot + GPQA Diamond eval functions ready.')
print()
print('Published reference points:')
print('  Jackrong Qwen3.5-4B-Claude distilled: GPQA Diamond 38.9%, ARC-25shot 66.4%')
print('  gpt-oss-20b published:                 GSM8K 68.9% (our 0-shot: 84.6%)')
print('  Qwen3-4B-Base published:               GSM8K 74.1% (5-shot)')


In [ ]:
# Step 3d: Run few-shot + GPQA Diamond evals
# These give us numbers comparable to published benchmarks

fewshot_results = {}

for model_name, model_path, _ in models_to_eval:
    print(f'\n{"="*60}')
    print(f'Few-shot eval: {model_name}')
    print(f'{"="*60}')
    
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path, max_seq_length=max_seq_length, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    
    results = {'model': model_name}
    
    # GSM8K 5-shot (standard published methodology)
    print('\n  GSM8K 5-shot...')
    results['gsm8k_5shot'] = eval_gsm8k_few_shot(model, tokenizer, n=500, n_shot=5)
    print(f'  GSM8K 5-shot: {results["gsm8k_5shot"]["accuracy"]}%')
    
    # GPQA Diamond 0-shot (for Jackrong comparison)
    print('\n  GPQA Diamond 0-shot...')
    results['gpqa_diamond'] = eval_gpqa_diamond(model, tokenizer)
    print(f'  GPQA Diamond: {results["gpqa_diamond"]["accuracy"]}%')
    
    fewshot_results[model_name] = results
    
    del model, tokenizer
    torch.cuda.empty_cache()

# Comparison table
print(f'\n{"="*60}')
print('FEW-SHOT + GPQA RESULTS (comparable to published benchmarks)')
print(f'{"="*60}')
print(f'{"Model":25s} {"GSM8K 0-shot":>14s} {"GSM8K 5-shot":>14s} {"GPQA Diamond":>14s}')
print('-' * 70)
for name in fewshot_results:
    zero = all_results.get(name, {}).get('gsm8k', {}).get('accuracy', '—')
    five = fewshot_results[name].get('gsm8k_5shot', {}).get('accuracy', '—')
    gpqa = fewshot_results[name].get('gpqa_diamond', {}).get('accuracy', '—')
    print(f'{name:25s} {str(zero)+"%":>14s} {str(five)+"%":>14s} {str(gpqa)+"%":>14s}')

print(f'\nPublished references:')
print(f'  Jackrong Qwen3.5-4B-Claude distilled: GPQA Diamond 38.9%')
print(f'  Qwen3-4B-Base:                         GSM8K 5-shot 74.1%')

with open(f'{DRIVE_DIR}/fewshot_results.json', 'w') as f:
    json.dump(fewshot_results, f, indent=2)

In [ ]:
# Step 4: Trick questions — side by side comparison

TRICK_QUESTIONS = [
    ('A farmer has 17 sheep. All but 9 die. How many sheep does the farmer have left?', '9'),
    ('If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?', '5 minutes'),
    ('A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost?', '$0.05'),
    ('A store offers a 20% discount on a jacket that costs $80. Then they add 8% sales tax. What is the final price?', '$69.12'),
    ('If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly?', 'No'),
]

trick_results = {}
for model_name, model_path, _ in models_to_eval:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_path, max_seq_length=max_seq_length, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    
    trick_results[model_name] = []
    for q, expected in TRICK_QUESTIONS:
        response = generate(model, tokenizer, q)
        trick_results[model_name].append({'problem': q, 'expected': expected, 'response': response[:500]})
    
    del model, tokenizer
    torch.cuda.empty_cache()

# Print side by side
for i, (q, expected) in enumerate(TRICK_QUESTIONS):
    print(f'\n{"="*70}')
    print(f'Q: {q}')
    print(f'Expected: {expected}')
    for model_name in trick_results:
        resp = trick_results[model_name][i]['response'][:300]
        print(f'\n  [{model_name}]')
        print(f'  {resp}')

with open(f'{DRIVE_DIR}/trick_results.json', 'w') as f:
    json.dump(trick_results, f, indent=2)

In [ ]:
# Step 5: Push best model to HuggingFace
from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))

# Pick best model based on eval results
best_name = max(all_results, key=lambda k: all_results[k].get('gsm8k', {}).get('accuracy', 0))
print(f'Best model: {best_name}')

best_dir = f'{DRIVE_DIR}/merged_{best_name.replace("distilled-", "")}'

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=best_dir, max_seq_length=max_seq_length, dtype=None, load_in_4bit=False,
)

# Push merged 16-bit
model.push_to_hub_merged(
    'bmeyer2025/qwen3.5-4b-glm5-reasoning-distilled',
    tokenizer,
    save_method='merged_16bit',
    token=userdata.get('HF_TOKEN'),
)
print('Pushed merged model to HuggingFace!')

In [ ]:
# Step 6: Export GGUF for local inference
model.push_to_hub_gguf(
    'bmeyer2025/qwen3.5-4b-glm5-reasoning-distilled',
    tokenizer,
    quantization_method=['q4_k_m', 'q8_0'],
    token=userdata.get('HF_TOKEN'),
)
print('GGUF files pushed to HuggingFace!')
print()
print('Run locally with:')
print('  ollama run hf.co/bmeyer2025/qwen3.5-4b-glm5-reasoning-distilled:Q4_K_M')

## Summary

After running this notebook, you'll have:

1. LoRA weights downloaded from Tinker
2. Merged full models saved to Google Drive
3. Eval results on clean benchmarks (GSM8K, MATH, ARC)
4. Side-by-side trick question comparison
5. Best model pushed to HuggingFace (merged 16-bit + GGUF)

Update the checkpoint paths above if you've done GRPO — swap the SFT
checkpoint paths for the GRPO checkpoint paths.